# Vocabulary Curation + Semantic Frame Assignment
Produces two artefacts:
- **`curated_vocabulary.json`** — three GPT-2 token-length-stratified pools (verbs, human nouns, force nouns)
- **`semantic_frames.json`** — per-verb-pair subsets of compatible patients and agents, assigned via LLM

All nouns are stored in **singular** form. Token lengths are computed with a leading space
(mid-sentence context, matching GPT-2 tokenisation).

**Phase 1** (cells 1–6): build and validate the vocabulary pools  
**Phase 2** (cells 7–9): assign semantic frames per verb pair via LLM


## 0 · Config
Edit these paths before running anything else.

In [1]:
import os

HANNA_CSV    = "../external/hannah_animacy/low_context_atypical_animacy_matched.csv"
OUTPUT_VOCAB = "../dataset/semantic_meaningful/curated_vocabulary.json"
OUTPUT_FRAMES = "../dataset/semantic_meaningful/semantic_frames.json"

# Set your OpenAI key here (Phase 2 only — leave empty to skip)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL = "gpt-4o"

print(f"Hanna CSV    : {os.path.abspath(HANNA_CSV)}")
print(f"Vocab output : {os.path.abspath(OUTPUT_VOCAB)}")
print(f"Frames output: {os.path.abspath(OUTPUT_FRAMES)}")
print(f"LLM step     : {'enabled' if OPENAI_API_KEY else 'DISABLED (no API key)'}")


Hanna CSV    : c:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\external\hannah_animacy\low_context_atypical_animacy_matched.csv
Vocab output : c:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\curated_vocabulary.json
Frames output: c:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\semantic_frames.json
LLM step     : enabled


## 1 · Imports

In [2]:
import json, re, time
from collections import defaultdict
from pathlib import Path

import pandas as pd
import inflect
import nltk
nltk.download("wordnet", quiet=True)
from nltk.corpus import wordnet as wn
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
inflect_p = inflect.engine()
print("GPT-2 tokenizer and inflect loaded.")

def tok_len(word: str) -> int:
    """Token count with a leading space (mid-sentence context)."""
    return len(tokenizer(" " + word)["input_ids"])

def group_by_tok(words: list) -> dict:
    d = defaultdict(list)
    for w in words:
        d[tok_len(w)].append(w)
    return dict(d)


GPT-2 tokenizer and inflect loaded.


## 2 · Seed Lists

In [3]:
ANIMATE_VERBS = [
    # Institutional / authority
    "hired", "fired", "promoted", "demoted", "recruited", "appointed",
    "elected", "nominated", "expelled", "escorted", "kidnapped",
    # Legal / judicial
    "arrested", "interrogated", "investigated", "prosecuted", "indicted",
    "convicted", "acquitted", "sentenced", "accused", "extradited", "pardoned",
    # Social / interpersonal
    "interviewed", "congratulated", "praised", "mocked", "greeted",
    "thanked", "scolded", "bribed", "blackmailed",
    # Medical / educational
    "vaccinated", "counseled", "educated", "advised", "instructed",
    "coached", "mentored", "tutored", "lectured", "consulted",
    # Commercial
    "purchased", "sold",
    # Emergency
    "rescued", "piloted",
]

INANIMATE_VERBS = [
    # Compression / impact
    "crushed", "smashed", "flattened", "compressed", "pounded", "battered",
    # Destruction
    "demolished", "shattered", "wrecked", "blasted", "obliterated",
    # Burial / inundation
    "buried", "submerged", "flooded", "engulfed", "inundated",
    # Thermal
    "scorched", "charred", "frozen",
    # Geological
    "eroded", "fractured", "cracked", "collapsed",
    # Motion / toppling
    "toppled", "warped", "dented", "ruptured",
]

# Written as plurals for readability — singularised in Step 1
FORCE_SEEDS_PLURAL = [
    # Meteorological
    "storms", "floods", "hurricanes", "earthquakes", "wildfires",
    "winds", "waves", "tides", "currents", "frost", "lightning",
    "tornadoes", "avalanches", "blizzards", "tsunamis",
    # Geological mass
    "boulders", "rocks", "stones", "glaciers", "icebergs",
    "landslides", "mudslides", "debris", "rubble",
    # Biological mass
    "trees", "branches", "logs",
    # Large vehicles / structures
    "trains", "trucks", "tractors", "bulldozers", "cranes", "ships",
    "walls", "ceilings", "roofs", "pillars",
]

print(f"Animate verbs  : {len(ANIMATE_VERBS)}")
print(f"Inanimate verbs: {len(INANIMATE_VERBS)}")
print(f"Force seeds    : {len(FORCE_SEEDS_PLURAL)} (before singularisation)")


Animate verbs  : 45
Inanimate verbs: 27
Force seeds    : 37 (before singularisation)


## Step 1 · Singularise Force Seeds
Seeds are written as plurals for readability but must be singularised before tokenisation.
Some nouns cross a GPT-2 token-length stratum when singularised (e.g. `tornadoes` → `tornado`: 2-tok → 1-tok).


In [4]:
def to_singular(word: str) -> str:
    sg = inflect_p.singular_noun(word)
    return sg if sg else word  # inflect returns False for already-singular / uncountable

FORCE_SEEDS = [to_singular(w) for w in FORCE_SEEDS_PLURAL]

print(f"{'Plural':<15} {'Singular':<15} pl-tok -> sg-tok")
print("-" * 50)
crossed = []
for pl, sg in zip(FORCE_SEEDS_PLURAL, FORCE_SEEDS):
    pl_l, sg_l = tok_len(pl), tok_len(sg)
    flag = " ← stratum change" if pl_l != sg_l else ""
    print(f"  {pl:<15} {sg:<15} {pl_l} -> {sg_l}{flag}")
    if pl_l != sg_l:
        crossed.append((pl, sg))

print(f"\n{len(crossed)} force nouns change token length when singularised.")
print(f"Final force seeds ({len(FORCE_SEEDS)}): {FORCE_SEEDS}")


Plural          Singular        pl-tok -> sg-tok
--------------------------------------------------
  storms          storm           1 -> 1
  floods          flood           1 -> 1
  hurricanes      hurricane       1 -> 1
  earthquakes     earthquake      1 -> 1
  wildfires       wildfire        1 -> 1
  winds           wind            1 -> 1
  waves           wave            1 -> 1
  tides           tide            1 -> 1
  currents        current         1 -> 1
  frost           frost           1 -> 1
  lightning       lightning       1 -> 1
  tornadoes       tornado         2 -> 1 ← stratum change
  avalanches      avalanche       2 -> 1 ← stratum change
  blizzards       blizzard        3 -> 2 ← stratum change
  tsunamis        tsunami         3 -> 1 ← stratum change
  boulders        boulder         2 -> 1 ← stratum change
  rocks           rock            1 -> 1
  stones          stone           1 -> 1
  glaciers        glacier         1 -> 1
  icebergs        iceberg         2 

## Step 2 · Extract Hanna Human Nouns
Already singular — used as-is. Multi-word entries (e.g. 'police officer') are excluded since they cannot fill a 1-token slot.

In [5]:
df = pd.read_csv(HANNA_CSV)
all_humans = sorted(df["human"].dropna().unique().tolist())
single_word_humans = [h for h in all_humans if " " not in h]
multi_word_humans  = [h for h in all_humans if " " in h]

print(f"Total unique human entries : {len(all_humans)}")
print(f"Single-word (usable)       : {len(single_word_humans)}")
print(f"Multi-word  (excluded)     : {len(multi_word_humans)}")
print(f"  -> {multi_word_humans}")


Total unique human entries : 127
Single-word (usable)       : 107
Multi-word  (excluded)     : 20
  -> ['art director', 'artistic director', 'assistant director', 'assistant professor', 'associate justice', 'associate professor', 'attorney general', 'change over', 'chief engineer', 'chief executive officer', 'chief of staff', 'city manager', 'executive director', 'executive officer', 'major general', 'police officer', 'prime minister', 'secretary of state', 'test pilot', 'vice president']


## Step 3 · GPT-2 Token Length Grouping
All three pools stratified by token count. The target stratum for this project is **1-token** (guarantees identical sentence length across all minimal pairs).

In [6]:
clean_by_tok   = group_by_tok(ANIMATE_VERBS)
corrupt_by_tok = group_by_tok(INANIMATE_VERBS)
humans_by_tok  = group_by_tok(single_word_humans)
forces_by_tok  = group_by_tok(FORCE_SEEDS)

print("Verb token-length strata:")
for n in sorted(set(list(clean_by_tok) + list(corrupt_by_tok))):
    c = clean_by_tok.get(n, [])
    k = corrupt_by_tok.get(n, [])
    print(f"  {n}-tok  clean={len(c):>3}  corrupt={len(k):>3}  pairs={len(c)*len(k):>4}")

print("\nHuman noun strata:")
for n, lst in sorted(humans_by_tok.items()):
    print(f"  {n}-tok: {len(lst):>3}  {sorted(lst)}")

print("\nForce noun strata:")
for n, lst in sorted(forces_by_tok.items()):
    print(f"  {n}-tok: {len(lst):>3}  {sorted(lst)}")


Verb token-length strata:
  1-tok  clean= 34  corrupt= 22  pairs= 748
  2-tok  clean= 11  corrupt=  5  pairs=  55

Human noun strata:
  1-tok:  82  ['accountant', 'administrator', 'agent', 'announcer', 'architect', 'assistant', 'associate', 'attorney', 'auditor', 'author', 'baker', 'bartender', 'biologist', 'boy', 'broadcaster', 'broker', 'butcher', 'chairman', 'chancellor', 'chemist', 'child', 'cleaner', 'clerk', 'coach', 'collector', 'consultant', 'cop', 'cousin', 'curator', 'dean', 'director', 'engineer', 'entrepreneur', 'father', 'fisher', 'founder', 'girl', 'grandfather', 'grandmother', 'grandson', 'husband', 'inspector', 'journalist', 'judge', 'lawyer', 'manager', 'mathematician', 'minister', 'mother', 'nephew', 'niece', 'nurse', 'operator', 'painter', 'partner', 'pastor', 'person', 'physician', 'pilot', 'planner', 'policeman', 'president', 'priest', 'producer', 'programmer', 'purchaser', 'reporter', 'researcher', 'sailor', 'scientist', 'scout', 'secretary', 'sheriff', 'student',

## Step 4 · WordNet Validation — Force Pool
Verifies each force noun is a descendant of `physical_entity.n.01` or `event.n.01`. Acts as a sanity check that no animate nouns slipped in.

In [7]:
_physical_entity = wn.synset("physical_entity.n.01")
_event           = wn.synset("event.n.01")

def is_physical_or_event(word: str):
    for synset in wn.synsets(word, pos=wn.NOUN):
        for path in synset.hypernym_paths():
            if _physical_entity in path or _event in path:
                return True, synset.name()
    return False, None

verified_forces, rejected_forces = [], []
for w in FORCE_SEEDS:
    ok, synset_name = is_physical_or_event(w)
    status = "OK  " if ok else "FAIL"
    print(f"  [{status}] {w:<15} {tok_len(w)}-tok  [{synset_name}]")
    (verified_forces if ok else rejected_forces).append(w)

print(f"\nVerified: {len(verified_forces)} | Rejected: {len(rejected_forces)}")
if rejected_forces:
    print(f"Rejected: {rejected_forces}")

forces_by_tok_final = group_by_tok(verified_forces)


  [OK  ] storm           1-tok  [storm.n.01]
  [OK  ] flood           1-tok  [flood.n.01]
  [OK  ] hurricane       1-tok  [hurricane.n.01]
  [OK  ] earthquake      1-tok  [earthquake.n.01]
  [OK  ] wildfire        1-tok  [wildfire.n.01]
  [OK  ] wind            1-tok  [wind.n.01]
  [OK  ] wave            1-tok  [wave.n.01]
  [OK  ] tide            1-tok  [tide.n.01]
  [OK  ] current         1-tok  [current.n.01]
  [OK  ] frost           1-tok  [frost.n.01]
  [OK  ] lightning       1-tok  [lightning.n.01]
  [OK  ] tornado         1-tok  [tornado.n.01]
  [OK  ] avalanche       1-tok  [avalanche.n.01]
  [OK  ] blizzard        2-tok  [blizzard.n.01]
  [OK  ] tsunami         1-tok  [tsunami.n.01]
  [OK  ] boulder         1-tok  [boulder.n.01]
  [OK  ] rock            1-tok  [rock.n.01]
  [OK  ] stone           1-tok  [rock.n.01]
  [OK  ] glacier         1-tok  [glacier.n.01]
  [OK  ] iceberg         1-tok  [iceberg.n.01]
  [OK  ] landslide       1-tok  [landslide.n.01]
  [OK  ] mudslide    

## Step 5 · Morphology Enrichment
Attaches plural form and determiner to each noun (stored as metadata — the template always uses `the`).

In [8]:
def get_morphology(word: str) -> dict:
    plural = inflect_p.plural(word)
    return {
        "word":        word,
        "plural":      plural,
        "determiner":  inflect_p.a(word).split()[0],  # reference only — use 'the' in templates
        "tok_len_sg":  tok_len(word),
        "tok_len_pl":  tok_len(plural),
    }

humans_enriched = [get_morphology(w) for w in single_word_humans]
forces_enriched = [get_morphology(w) for w in verified_forces]

# Flag plurals that cross a token-length stratum (relevant if plural forms are ever needed)
pl_crossings = [r for r in humans_enriched if r["tok_len_sg"] != r["tok_len_pl"]]
print(f"Human nouns whose plural crosses a stratum: {len(pl_crossings)}")
for r in pl_crossings[:8]:
    print(f"  {r['word']:<20} sg={r['tok_len_sg']}  pl={r['tok_len_pl']}")


Human nouns whose plural crosses a stratum: 31
  accountant           sg=1  pl=2
  anchorman            sg=2  pl=3
  announcer            sg=1  pl=2
  archaeologist        sg=2  pl=1
  archivist            sg=2  pl=3
  auditor              sg=1  pl=2
  baker                sg=1  pl=2
  bartender            sg=1  pl=2


## Step 6 · Assemble & Export `curated_vocabulary.json`

In [9]:
vocab = {
    "meta": {
        "description":                  "Phase 1 curated vocabulary — verb_agent_selection_passive",
        "hanna_csv":                    str(HANNA_CSV),
        "token_lengths_use_space_prefix": True,
        "number":                       "singular — all nouns stored in singular form",
        "determiner_policy":            "always use 'the' in templates — ignore stored determiner field",
    },
    "verbs": {
        "note": "Clean/corrupt pairs must share the same token-length stratum",
        "by_token_length": {
            str(n): {
                "clean":   sorted(clean_by_tok.get(n, [])),
                "corrupt": sorted(corrupt_by_tok.get(n, [])),
                "n_pairs": len(clean_by_tok.get(n, [])) * len(corrupt_by_tok.get(n, [])),
            }
            for n in sorted(set(list(clean_by_tok) + list(corrupt_by_tok)))
        },
    },
    "human_pool": {
        "note":               "PATIENT slot + ANIMATE AGENT slot — singular forms",
        "source":             "Hanna et al. (2022) — human column",
        "multi_word_excluded": multi_word_humans,
        "by_token_length": {
            str(n): {
                "words":      sorted(lst),
                "morphology": [r for r in humans_enriched if r["tok_len_sg"] == n],
            }
            for n, lst in sorted(humans_by_tok.items())
        },
    },
    "force_pool": {
        "note":               "INANIMATE AGENT slot — singular forms, WordNet-verified",
        "source":             "Manually curated, singularised via inflect, verified via WordNet",
        "rejected_by_wordnet": rejected_forces,
        "by_token_length": {
            str(n): {
                "words":      sorted(lst),
                "morphology": [r for r in forces_enriched if r["tok_len_sg"] == n],
            }
            for n, lst in sorted(forces_by_tok_final.items())
        },
    },
}

Path(OUTPUT_VOCAB).parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_VOCAB, "w") as f:
    json.dump(vocab, f, indent=2)

total_pairs = sum(v["n_pairs"] for v in vocab["verbs"]["by_token_length"].values())
v1 = vocab["verbs"]["by_token_length"].get("1", {})
h1 = len(vocab["human_pool"]["by_token_length"].get("1", {}).get("words", []))
f1 = len(vocab["force_pool"]["by_token_length"].get("1", {}).get("words", []))

print(f"Saved -> {Path(OUTPUT_VOCAB).resolve()}")
print(f"Verb pairs (all strata) : {total_pairs:,}")
print(f"Human nouns (1-tok)     : {h1}")
print(f"Force nouns (1-tok)     : {f1}")
print(f"Upper bound (1-tok pairs) : {v1.get('n_pairs',0) * h1 * f1:,} combinatorial sentences")
print("\nPhase 1 complete.")


Saved -> C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\curated_vocabulary.json
Verb pairs (all strata) : 803
Human nouns (1-tok)     : 82
Force nouns (1-tok)     : 34
Upper bound (1-tok pairs) : 2,085,424 combinatorial sentences

Phase 1 complete.


---
# Phase 2 · Semantic Frame Assignment

For each verb pair `(animate_verb, inanimate_verb)` we ask an LLM to select
semantically compatible subsets from the **existing 1-token pools** (no new words are generated).
The output is a list of *semantic frames*, each containing:

```
{
  "animate_verb":    "hired",
  "inanimate_verb":  "demolished",
  "patients":        ["worker", "contractor", ...],   # from human_pool (1-tok)
  "animate_agents":  ["manager", "director", ...],   # from human_pool (1-tok)
  "inanimate_agents":["storm", "bulldozer", ...],   # from force_pool (1-tok)
}
```

The LLM is only a *selector* — every word in its output must appear in the input pools.
An automatic post-hoc filter rejects any word that slipped out of the pools.


## Step 7 · Build 1-Token Verb Pairs
We restrict to the 1-token stratum to guarantee uniform sentence length across all minimal pairs.

In [12]:
# Reload the vocabulary (works even if kernel was restarted after Phase 1)
with open(OUTPUT_VOCAB) as f:
    vocab = json.load(f)

clean_1tok   = vocab["verbs"]["by_token_length"]["1"]["clean"]
corrupt_1tok = vocab["verbs"]["by_token_length"]["1"]["corrupt"]
human_1tok   = vocab["human_pool"]["by_token_length"]["1"]["words"]
force_1tok   = vocab["force_pool"]["by_token_length"]["1"]["words"]

# Verb pairs: every clean × corrupt combination is valid at the token level.
# We form a *curated* list of pairs that are semantically coherent
# (same broad event domain). Extend / trim this list as needed.
VERB_PAIRS = [
    # Institutional
    ("hired",        "demolished"),
    ("fired",        "crushed"),
    ("promoted",     "flooded"),
    ("recruited",    "buried"),
    ("appointed",    "collapsed"),
    ("elected",      "toppled"),
    ("expelled",     "shattered"),
    # Legal
    ("arrested",     "smashed"),
    ("prosecuted",   "wrecked"),
    ("convicted",    "frozen"),
    ("accused",      "battered"),
    ("indicted",     "cracked"),
    # Social
    ("praised",      "flattened"),
    ("mocked",       "warped"),
    ("praised",  "cracked"),
    ("thanked",  "charred"),
    # Medical / educational
    ("vaccinated",   "engulfed"),
    ("educated",     "eroded"),
    ("coached",      "charred"),
    # Commercial
    ("purchased",    "crushed"),
    ("sold",         "flooded"),
    # Emergency
    ("rescued",      "submerged"),
]

print(f"Total verb pairs defined : {len(VERB_PAIRS)}")
print(f"Clean  pool (1-tok)      : {len(clean_1tok)} verbs")
print(f"Corrupt pool (1-tok)     : {len(corrupt_1tok)} verbs")
print(f"Human  pool (1-tok)      : {len(human_1tok)} nouns")
print(f"Force  pool (1-tok)      : {len(force_1tok)} nouns")
print()

# Sanity check: all paired verbs are in the 1-tok pool
missing = [(a, i) for a, i in VERB_PAIRS if a not in clean_1tok or i not in corrupt_1tok]
if missing:
    print(f"WARNING — these pairs contain out-of-pool verbs: {missing}")
else:
    print("All verb pairs are within the 1-token pools. ✓")


Total verb pairs defined : 22
Clean  pool (1-tok)      : 34 verbs
Corrupt pool (1-tok)     : 22 verbs
Human  pool (1-tok)      : 82 nouns
Force  pool (1-tok)      : 34 nouns

All verb pairs are within the 1-token pools. ✓


## Step 8 · LLM Semantic Frame Assignment

For each verb pair the LLM is given both 1-token pools and asked to return
JSON with three keys: `patients`, `animate_agents`, `inanimate_agents`.

A post-hoc filter removes any hallucinated words (words not in the pools).
Set `OPENAI_API_KEY` in cell 0 to run this step; otherwise a cached
`semantic_frames.json` from a previous run is loaded if available.


In [14]:
from dotenv import load_dotenv
load_dotenv()  # reads .env from the current working directory

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
print(f"API key loaded: {'✓' if OPENAI_API_KEY else '✗ NOT FOUND'}")

API key loaded: ✓


In [19]:
def build_prompt(animate_v, inanimate_v, human_pool, force_pool):
    return f"""You are a linguist building a psycholinguistics dataset.
Template: "The [PATIENT] was [VERB] by the [AGENT]."

Verb pair:
  animate verb   (clean)   : '{animate_v}'
  inanimate verb (corrupt)  : '{inanimate_v}'

Available HUMAN nouns (for PATIENT and ANIMATE AGENT slots):
{human_pool}

Available FORCE nouns (for INANIMATE AGENT slot):
{force_pool}

Task: select subsets that make BOTH sentences individually plausible.
- patients       : nouns from the human list that are plausible patients of BOTH verbs
- animate_agents : nouns from the human list that are plausible agents of '{animate_v}'
- inanimate_agents: nouns from the force list that are plausible agents of '{inanimate_v}'

Constraints:
- Return ONLY words that appear verbatim in the lists above (no new words)
- Return 5-12 items per key where possible
- Return valid JSON only, no commentary

{{
  "patients": [...],
  "animate_agents": [...],
  "inanimate_agents": [...]
}}"""


def query_llm(prompt: str, model: str, api_key: str) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return response.choices[0].message.content


def filter_to_pool(words: list, pool: set, max_items: int = 10) -> list:
    filtered = [w for w in words if w in pool]
    hallucinated = [w for w in words if w not in pool]
    if hallucinated:
        print(f"    ⚠ removed (not in pool): {hallucinated}")
    return filtered[:max_items]  # hard cap regardless of prompt compliance


human_set = set(human_1tok)
force_set  = set(force_1tok)

human_pool_str = json.dumps(human_1tok)
force_pool_str = json.dumps(force_1tok)

frames = []

if OPENAI_API_KEY:
    print(f"Running LLM assignment for {len(VERB_PAIRS)} verb pairs...\n")
    for animate_v, inanimate_v in VERB_PAIRS:
        print(f"  {animate_v} / {inanimate_v}")
        prompt = build_prompt(animate_v, inanimate_v, human_pool_str, force_pool_str)
        try:
            raw = query_llm(prompt, LLM_MODEL, OPENAI_API_KEY)
            data = json.loads(raw)
            frame = {
                "animate_verb":     animate_v,
                "inanimate_verb":   inanimate_v,
                "patients":         filter_to_pool(data.get("patients", []),         human_set),
                "animate_agents":   filter_to_pool(data.get("animate_agents", []),   human_set),
                "inanimate_agents": filter_to_pool(data.get("inanimate_agents", []), force_set),
            }
            n = len(frame["patients"]) * len(frame["animate_agents"]) * len(frame["inanimate_agents"])
            print(f"    patients={len(frame['patients'])}  anim_agents={len(frame['animate_agents'])}  inanim_agents={len(frame['inanimate_agents'])}  → {n:,} pairs")
            frames.append(frame)
            time.sleep(0.5)  # be nice to the API
        except Exception as e:
            print(f"    ERROR: {e}")
else:
    print("No API key set — skipping LLM calls.")
    if Path(OUTPUT_FRAMES).exists():
        with open(OUTPUT_FRAMES) as f:
            frames = json.load(f)["frames"]
        print(f"Loaded {len(frames)} frames from existing {OUTPUT_FRAMES}")
    else:
        print("No cached frames found. Set OPENAI_API_KEY in cell 0 and re-run.")


Running LLM assignment for 22 verb pairs...

  hired / demolished
    patients=10  anim_agents=10  inanim_agents=10  → 1,000 pairs
  fired / crushed
    patients=10  anim_agents=10  inanim_agents=10  → 1,000 pairs
  promoted / flooded
    ⚠ removed (not in pool): ['worker']
    patients=9  anim_agents=10  inanim_agents=10  → 900 pairs
  recruited / buried
    patients=10  anim_agents=10  inanim_agents=10  → 1,000 pairs
  appointed / collapsed
    patients=7  anim_agents=6  inanim_agents=4  → 168 pairs
  elected / toppled
    patients=10  anim_agents=10  inanim_agents=10  → 1,000 pairs
  expelled / shattered
    patients=10  anim_agents=8  inanim_agents=9  → 720 pairs
  arrested / smashed
    patients=10  anim_agents=6  inanim_agents=10  → 600 pairs
  prosecuted / wrecked
    ⚠ removed (not in pool): ['prosecutor']
    patients=10  anim_agents=6  inanim_agents=10  → 600 pairs
  convicted / frozen
    ⚠ removed (not in pool): ['prosecutor', 'officer', 'magistrate']
    ⚠ removed (not in 

## Step 9 · Export `semantic_frames.json` & Coverage Report

In [20]:
if not frames:
    print("No frames to export — run Step 8 first.")
else:
    total_pairs = sum(
        len(f["patients"]) * len(f["animate_agents"]) * len(f["inanimate_agents"])
        for f in frames
    )

    output = {
        "meta": {
            "description":  "Phase 2 semantic frames — per-verb-pair compatible subsets",
            "vocab_source": str(OUTPUT_VOCAB),
            "llm_model":    LLM_MODEL,
            "n_verb_pairs":  len(frames),
            "total_raw_pairs": total_pairs,
            "note": "All words come verbatim from curated_vocabulary.json (1-tok pools)",
        },
        "frames": frames,
    }

    Path(OUTPUT_FRAMES).parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_FRAMES, "w") as f:
        json.dump(output, f, indent=2)

    print(f"Saved -> {Path(OUTPUT_FRAMES).resolve()}")
    print(f"\n{'Verb pair':<35} {'P':>4} {'Aa':>4} {'Ai':>4} {'pairs':>8}")
    print("-" * 60)
    for fr in frames:
        p  = len(fr["patients"])
        aa = len(fr["animate_agents"])
        ai = len(fr["inanimate_agents"])
        print(f"  {fr['animate_verb']:<16} / {fr['inanimate_verb']:<14} {p:>4} {aa:>4} {ai:>4} {p*aa*ai:>8,}")
    print("-" * 60)
    print(f"  {'TOTAL':<33} {total_pairs:>8,} raw pairs")
    print(f"\nTarget 10k after Phase 4 filtering: {'✓ likely sufficient' if total_pairs >= 12000 else '⚠ may need more verb pairs'}")


Saved -> C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\semantic_frames.json

Verb pair                              P   Aa   Ai    pairs
------------------------------------------------------------
  hired            / demolished       10   10   10    1,000
  fired            / crushed          10   10   10    1,000
  promoted         / flooded           9   10   10      900
  recruited        / buried           10   10   10    1,000
  appointed        / collapsed         7    6    4      168
  elected          / toppled          10   10   10    1,000
  expelled         / shattered        10    8    9      720
  arrested         / smashed          10    6   10      600
  prosecuted       / wrecked          10    6   10      600
  convicted        / frozen           10    7    7      490
  accused          / battered         10   10   10    1,000
  indicted         / cracked          10    8   10     

# Phase 3 — Minimal Pair Generator

Reads `semantic_frames.json` and produces `raw_pairs.jsonl` — the full
combinatorial expansion of all frames, before Phase 4 filtering.

Each line is one minimal pair:
```json
{
  "clean":          "The architect was hired by the director.",
  "corrupt":        "The architect was demolished by the avalanche.",
  "patient":        "architect",
  "animate_verb":   "hired",
  "inanimate_verb": "demolished",
  "animate_agent":  "director",
  "inanimate_agent":"avalanche",
  "frame":          "hired/demolished",
  "uid":            "a3f9c1..."
}
```


## 0 · Config

In [1]:
INPUT_FRAMES = "../dataset/semantic_meaningful/semantic_frames.json"
OUTPUT_JSONL  = "../dataset/semantic_meaningful/raw_pairs.jsonl"

# Hard cap on pairs per frame — prevents any single frame dominating
MAX_PAIRS_PER_FRAME = 1000

print(f"Frames : {INPUT_FRAMES}")
print(f"Output : {OUTPUT_JSONL}")
print(f"Cap/frame: {MAX_PAIRS_PER_FRAME}")


Frames : ../dataset/semantic_meaningful/semantic_frames.json
Output : ../dataset/semantic_meaningful/raw_pairs.jsonl
Cap/frame: 1000


## 1 · Imports

In [2]:
import json, uuid, random
from pathlib import Path
from itertools import product
from collections import defaultdict

random.seed(42)


## 2 · Load & Validate Frames

In [3]:
with open(INPUT_FRAMES) as f:
    data = json.load(f)

frames = data["frames"]
print(f"Loaded {len(frames)} frames from {INPUT_FRAMES}")
print()

# Pre-flight checks
issues = []
for fr in frames:
    av, iv = fr["animate_verb"], fr["inanimate_verb"]
    # Patient must not overlap with animate_agent (would generate "X was V by X")
    overlap = set(fr["patients"]) & set(fr["animate_agents"])
    if overlap:
        issues.append(f"  [{av}/{iv}] patient ∩ animate_agent overlap: {sorted(overlap)}")
    # Minimum size check
    if len(fr["patients"]) < 3 or len(fr["animate_agents"]) < 3 or len(fr["inanimate_agents"]) < 3:
        issues.append(f"  [{av}/{iv}] underpowered slot (< 3 words)")

if issues:
    print("⚠ Pre-flight warnings:")
    for i in issues:
        print(i)
else:
    print("Pre-flight checks passed. ✓")


Loaded 15 frames from ../dataset/semantic_meaningful/semantic_frames.json

⚠ Pre-flight warnings:
  [hired/demolished] patient ∩ animate_agent overlap: ['manager']
  [fired/crushed] patient ∩ animate_agent overlap: ['administrator']
  [promoted/flooded] patient ∩ animate_agent overlap: ['administrator', 'manager']
  [recruited/buried] patient ∩ animate_agent overlap: ['administrator']
  [elected/toppled] patient ∩ animate_agent overlap: ['administrator', 'chairman', 'chancellor', 'dean', 'director', 'manager', 'minister', 'president', 'superintendent', 'teacher']
  [indicted/cracked] patient ∩ animate_agent overlap: ['agent', 'auditor']
  [praised/flattened] patient ∩ animate_agent overlap: ['administrator', 'architect', 'author']
  [mocked/warped] patient ∩ animate_agent overlap: ['author', 'broadcaster']
  [thanked/charred] patient ∩ animate_agent overlap: ['administrator', 'agent', 'architect', 'assistant', 'associate', 'attorney', 'auditor', 'author', 'baker']
  [vaccinated/engulfe

## 3 · Generate Minimal Pairs

Full combinatorial expansion, then:
1. Remove pairs where `patient == animate_agent` (same token in both slots)
2. Shuffle within each frame
3. Cap at `MAX_PAIRS_PER_FRAME` to ensure balanced frame representation


In [4]:
all_pairs = []
frame_counts = {}

for fr in frames:
    av    = fr["animate_verb"]
    iv    = fr["inanimate_verb"]
    key   = f"{av}/{iv}"

    frame_pairs = []
    for patient, aa, ia in product(fr["patients"], fr["animate_agents"], fr["inanimate_agents"]):
        if patient == aa:      # skip "The director was hired by the director."
            continue
        frame_pairs.append({
            "clean":           f"The {patient} was {av} by the {aa}.",
            "corrupt":         f"The {patient} was {iv} by the {ia}.",
            "patient":          patient,
            "animate_verb":     av,
            "inanimate_verb":   iv,
            "animate_agent":    aa,
            "inanimate_agent":  ia,
            "frame":            key,
            "uid":              uuid.uuid4().hex,
        })

    random.shuffle(frame_pairs)
    frame_pairs = frame_pairs[:MAX_PAIRS_PER_FRAME]
    frame_counts[key] = len(frame_pairs)
    all_pairs.extend(frame_pairs)

print(f"{'Frame':<35} {'pairs':>6}")
print("-" * 45)
for key, n in frame_counts.items():
    print(f"  {key:<33} {n:>6,}")
print("-" * 45)
print(f"  {'TOTAL':<33} {len(all_pairs):>6,}")


Frame                                pairs
---------------------------------------------
  hired/demolished                     990
  fired/crushed                        990
  promoted/flooded                     880
  recruited/buried                     490
  elected/toppled                      900
  expelled/shattered                   720
  arrested/smashed                     600
  prosecuted/wrecked                   600
  convicted/frozen                     490
  accused/battered                   1,000
  indicted/cracked                     780
  praised/flattened                    970
  mocked/warped                        980
  thanked/charred                      910
  vaccinated/engulfed                  880
---------------------------------------------
  TOTAL                             12,180


## 4 · Sanity Checks

In [5]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

def tok_count(sentence: str) -> int:
    return len(tokenizer(sentence)["input_ids"])

# Check all sentences have the same token length
lengths = set(tok_count(p["clean"]) for p in all_pairs)
lengths |= set(tok_count(p["corrupt"]) for p in all_pairs)

print(f"Distinct sentence token lengths: {sorted(lengths)}")
if len(lengths) == 1:
    print(f"All sentences are {list(lengths)[0]} tokens. ✓")
else:
    print("⚠ Token length mismatch — check verb/noun token lengths in vocabulary.")

# Verify no patient == animate_agent
same_slot = [p for p in all_pairs if p["patient"] == p["animate_agent"]]
print(f"patient == animate_agent collisions: {len(same_slot)} (should be 0)")

# Frame balance check
counts = list(frame_counts.values())
print(f"Pairs per frame — min: {min(counts)}  max: {max(counts)}  mean: {sum(counts)/len(counts):.0f}")


Distinct sentence token lengths: [8]
All sentences are 8 tokens. ✓
patient == animate_agent collisions: 0 (should be 0)
Pairs per frame — min: 490  max: 1000  mean: 812


## 5 · Export `raw_pairs.jsonl`

In [6]:
Path(OUTPUT_JSONL).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_JSONL, "w") as f:
    for pair in all_pairs:
        f.write(json.dumps(pair) + "\n")

size_kb = Path(OUTPUT_JSONL).stat().st_size / 1024
print(f"Saved {len(all_pairs):,} pairs -> {Path(OUTPUT_JSONL).resolve()}")
print(f"File size: {size_kb:.1f} KB")
print(f"\nReady for Phase 4 (plausibility filtering).")


Saved 12,180 pairs -> C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\raw_pairs.jsonl
File size: 3833.0 KB

Ready for Phase 4 (plausibility filtering).


In [7]:
Path(OUTPUT_JSONL).parent.mkdir(parents=True, exist_ok=True)

# Original (frame-grouped) order
with open(OUTPUT_JSONL, "w") as f:
    for pair in all_pairs:
        f.write(json.dumps(pair) + "\n")

# Shuffled version — frame labels are interleaved, better for batched inference
shuffled = all_pairs.copy()
random.shuffle(shuffled)

OUTPUT_JSONL_SHUFFLED = OUTPUT_JSONL.replace(".jsonl", "_shuffled.jsonl")
with open(OUTPUT_JSONL_SHUFFLED, "w") as f:
    for pair in shuffled:
        f.write(json.dumps(pair) + "\n")

size_kb          = Path(OUTPUT_JSONL).stat().st_size / 1024
size_kb_shuffled = Path(OUTPUT_JSONL_SHUFFLED).stat().st_size / 1024
print(f"Saved {len(all_pairs):,} pairs          -> {Path(OUTPUT_JSONL).resolve()} ({size_kb:.1f} KB)")
print(f"Saved {len(shuffled):,} pairs (shuffled) -> {Path(OUTPUT_JSONL_SHUFFLED).resolve()} ({size_kb_shuffled:.1f} KB)")
print(f"\nReady for Phase 4 (plausibility filtering).")

Saved 12,180 pairs          -> C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\raw_pairs.jsonl (3833.0 KB)
Saved 12,180 pairs (shuffled) -> C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\raw_pairs_shuffled.jsonl (3833.0 KB)

Ready for Phase 4 (plausibility filtering).


Phase 4

In [26]:
import json, os, time, hashlib
from pathlib import Path
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

INPUT_JSONL          = OUTPUT_JSONL_SHUFFLED
OUTPUT_KEPT          = "../dataset/semantic_meaningful/phase4_filtered.jsonl"
OUTPUT_REJECTED      = "../dataset/semantic_meaningful/phase4_rejected.jsonl"
CACHE_FILE           = "../dataset/semantic_meaningful/phase4_score_cache.jsonl"

CHEAP_MODEL          = "gpt-4o-mini"
STRONG_MODEL         = "gpt-4o"
BATCH_SIZE           = 25
CLEAN_PASS_MIN       = 4
CORRUPT_PASS_MAX     = 2
BORDERLINE_RECHECK   = True   # re-score score==3 cases with strong model

In [11]:
with open(INPUT_JSONL) as f:
    pairs = [json.loads(l) for l in f]

# Resume-safe cache: uid -> {clean_score, corrupt_score, model}
cache = {}
if Path(CACHE_FILE).exists():
    with open(CACHE_FILE) as f:
        for line in f:
            entry = json.loads(line)
            cache[entry["uid"]] = entry

print(f"Loaded {len(pairs):,} pairs | Cache hits: {len(cache):,}")

Loaded 12,180 pairs | Cache hits: 0


In [18]:
SYSTEM_PROMPT = """You are a plausibility judge for English sentences.
For each sentence, rate how plausible it is as a description of a real-world event.
Focus on whether the agent could realistically perform this action on this patient.
Ignore grammar — all sentences are grammatical.
Score: 1 = impossible, 2 = very odd, 3 = marginal, 4 = plausible, 5 = completely natural.
Respond with a JSON array, one object per sentence, in the same order:
[{"score": <int>, "reason": "<one sentence>"}, ...]"""

def score_sentences(sentences: list[str], model: str) -> list[dict]:
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Rate these sentences:\n{numbered}\nReturn a JSON object with key 'results' containing the array. The array MUST have exactly {len(sentences)} elements, one per sentence, in order."}
        ]
    )
    results = json.loads(response.choices[0].message.content)["results"]

    if len(results) != len(sentences):
        # Split batch in half and recurse instead of going 1-by-1
        if len(sentences) == 1:
            raise ValueError(f"Single sentence still failed: {sentences[0]}")
        mid = len(sentences) // 2
        results = score_sentences(sentences[:mid], model) + score_sentences(sentences[mid:], model)

    return results


def score_batch(batch: list[dict], model: str) -> list[dict]:
    """Score a batch of pairs; returns list of {uid, clean_score, corrupt_score, model}."""
    clean_sentences   = [p["clean"]   for p in batch]
    corrupt_sentences = [p["corrupt"] for p in batch]

    clean_scores   = score_sentences(clean_sentences,   model)
    corrupt_scores = score_sentences(corrupt_sentences, model)

    return [
        {
            "uid":           p["uid"],
            "clean_score":   clean_scores[i]["score"],
            "clean_reason":  clean_scores[i]["reason"],
            "corrupt_score": corrupt_scores[i]["score"],
            "corrupt_reason":corrupt_scores[i]["reason"],
            "model":         model,
        }
        for i, p in enumerate(batch)
    ]

In [21]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

Path(CACHE_FILE).parent.mkdir(parents=True, exist_ok=True)
cache_fh = open(CACHE_FILE, "a")

to_score = [p for p in pairs if p["uid"] not in cache]
print(f"Scoring {len(to_score):,} pairs with {CHEAP_MODEL}...")

def score_batch_parallel(batch: list[dict]) -> list[dict]:
    """Score clean and corrupt sentences concurrently."""
    clean_sentences   = [p["clean"]   for p in batch]
    corrupt_sentences = [p["corrupt"] for p in batch]

    with ThreadPoolExecutor(max_workers=2) as ex:
        f_clean   = ex.submit(score_sentences, clean_sentences,   CHEAP_MODEL)
        f_corrupt = ex.submit(score_sentences, corrupt_sentences, CHEAP_MODEL)
        clean_scores   = f_clean.result()
        corrupt_scores = f_corrupt.result()

    return [
        {
            "uid":           p["uid"],
            "clean_score":   clean_scores[i]["score"],
            "clean_reason":  clean_scores[i]["reason"],
            "corrupt_score": corrupt_scores[i]["score"],
            "corrupt_reason":corrupt_scores[i]["reason"],
            "model":         CHEAP_MODEL,
        }
        for i, p in enumerate(batch)
    ]

batches = [to_score[i : i + BATCH_SIZE] for i in range(0, len(to_score), BATCH_SIZE)]

CONCURRENT_BATCHES = 5   # number of batches in flight simultaneously

with tqdm(total=len(batches), unit="batch", desc="Phase 4 pass 1") as pbar:
    with ThreadPoolExecutor(max_workers=CONCURRENT_BATCHES) as pool:
        futures = {pool.submit(score_batch_parallel, b): b for b in batches}
        for future in as_completed(futures):
            try:
                results = future.result()
                for r in results:
                    cache[r["uid"]] = r
                    cache_fh.write(json.dumps(r) + "\n")
                cache_fh.flush()
            except Exception as e:
                tqdm.write(f"  batch failed: {e}")
            pbar.update(1)

cache_fh.close()
print("Pass 1 complete.")

Scoring 11,880 pairs with gpt-4o-mini...


Phase 4 pass 1: 100%|██████████| 476/476 [20:39<00:00,  2.60s/batch]

Pass 1 complete.


In [27]:
kept, rejected = [], []

for p in pairs:
    r = cache.get(p["uid"])
    if r is None:
        continue
    augmented = {
        **p,
        "clean_score":   r["clean_score"],
        "clean_reason":  r["clean_reason"],
        "corrupt_score": r["corrupt_score"],
        "corrupt_reason":r["corrupt_reason"],
        "phase4_model":  r["model"],
    }
    passes = (
        r["clean_score"]   >= CLEAN_PASS_MIN and
        r["corrupt_score"] <= CORRUPT_PASS_MAX and
        (r["clean_score"] - r["corrupt_score"]) >= 2
    )
    (kept if passes else rejected).append(augmented)

Path(OUTPUT_KEPT).parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_KEPT, "w") as f:
    for p in kept:     f.write(json.dumps(p) + "\n")
with open(OUTPUT_REJECTED, "w") as f:
    for p in rejected: f.write(json.dumps(p) + "\n")

print(f"Kept:     {len(kept):,}  ({100*len(kept)/len(pairs):.1f}%)")
print(f"Rejected: {len(rejected):,}  ({100*len(rejected)/len(pairs):.1f}%)")

Kept:     3,733  (30.6%)
Rejected: 8,447  (69.4%)


In [28]:
import collections

reasons = collections.Counter()
clean_scores_dist  = collections.Counter()
corrupt_scores_dist = collections.Counter()

for p in pairs:
    r = cache.get(p["uid"])
    if r is None:
        continue
    clean_scores_dist[r["clean_score"]]   += 1
    corrupt_scores_dist[r["corrupt_score"]] += 1

    if r["clean_score"] < CLEAN_PASS_MIN:
        reasons["clean too low"] += 1
    if r["corrupt_score"] > CORRUPT_PASS_MAX:
        reasons["corrupt too high"] += 1
    if (r["clean_score"] - r["corrupt_score"]) < 2:
        reasons["gap too small"] += 1

print("Rejection reasons (can overlap):")
for k, v in reasons.most_common():
    print(f"  {k}: {v:,}")

print("\nClean score distribution:")
for s in sorted(clean_scores_dist):
    print(f"  {s}: {clean_scores_dist[s]:,}")

print("\nCorrupt score distribution:")
for s in sorted(corrupt_scores_dist):
    print(f"  {s}: {corrupt_scores_dist[s]:,}")

Rejection reasons (can overlap):
  gap too small: 8,045
  clean too low: 5,609
  corrupt too high: 5,459

Clean score distribution:
  1: 105
  2: 3,482
  3: 2,022
  4: 5,977
  5: 594

Corrupt score distribution:
  1: 2,403
  2: 4,318
  3: 1,340
  4: 4,067
  5: 52


In [ ]:
if BORDERLINE_RECHECK:
    borderline_uids = {
        uid for uid, r in cache.items()
        if r["clean_score"] == 3 or r["corrupt_score"] == 3
    }
    borderline_pairs = [p for p in pairs if p["uid"] in borderline_uids]
    print(f"Re-scoring {len(borderline_pairs):,} borderline pairs with {STRONG_MODEL}...")

    cache_fh = open(CACHE_FILE, "a")
    for i in range(0, len(borderline_pairs), BATCH_SIZE):
        batch = borderline_pairs[i : i + BATCH_SIZE]
        try:
            results = score_batch(batch, STRONG_MODEL)
            for r in results:
                cache[r["uid"]] = r          # overwrite with stronger judgment
                cache_fh.write(json.dumps(r) + "\n")
            cache_fh.flush()
            time.sleep(0.5)
        except Exception as e:
            print(f"  ERROR: {e} — retrying in 10s")
            time.sleep(10)
    cache_fh.close()
    print("Pass 2 complete.")

In [22]:
from collections import Counter

rejection_by_frame = Counter(p["frame"] for p in rejected)
total_by_frame     = Counter(p["frame"] for p in pairs)

print(f"{'Frame':<30} {'Rejected':>8} {'Total':>8} {'Rej%':>7}")
print("-" * 58)
for frame, total in sorted(total_by_frame.items()):
    n_rej = rejection_by_frame.get(frame, 0)
    print(f"{frame:<30} {n_rej:>8,} {total:>8,} {100*n_rej/total:>6.1f}%")

NameError: name 'rejected' is not defined